In [2]:
import pickle # Load refs and annotations
import json
import os
import pandas as pd
import numpy as np
import pprint
import json
import cv2
import random

from typing import Any, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.utils.tensorboard import SummaryWriter

import torchvision
import torchvision.transforms as transforms
from torchvision.utils import draw_bounding_boxes
from torchvision import models
import torchmetrics

import pytorch_lightning as pl
from pytorch_lightning.utilities.types import STEP_OUTPUT

from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import CLIPProcessor, CLIPModel

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import clip
from ultralytics import YOLO
from PIL import Image, ImageDraw

from ipywidgets import FloatProgress
import math 
from torch.nn.modules.batchnorm import _BatchNorm

In [3]:
device = torch.device('cuda')

In [4]:
clip_model, clip_preprocess = clip.load("RN50", device=device)

In [5]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_area = max(0, x2 - x1 + 1) * max(0, y2 - y1 + 1) # +1 to avoid max(0,0) therefore avoiding 

    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union_area = box1_area + box2_area - intersection_area

    return intersection_area / union_area

In [8]:
class MetricMeter:
    def __init__(self, name="Default", threshold = 0.5, log_dir='./logs/default_run'):
        self.name = name
        self.metrics = []
        self.epochs = []
        self.threshold = threshold
        self.reset()
        self.best_cases_iou = []  # To store the best cases
        self.worst_cases_iou = []
        self.best_cases_sim = []  # To store the best cases
        self.worst_cases_sim = []
        self.writer = SummaryWriter(log_dir=log_dir)

# TODO add SummaryWriter to plot

    def reset(self):
        self.count = 0
        self.iou = 0
        self.epoch = 0
        self.correct_bboxes, self.overall = 0,0
        self.semantic = 0
        self.metrics = []
    
    def add_best_iou(self, item):
        if len(self.best_cases_iou) < 5:
            self.best_cases_iou.append(item)
        else:
            min_best_case = min(self.best_cases_iou, key=lambda x: (x['iou']))
            if item['iou'] > min_best_case['iou']:
                self.best_cases_iou.remove(min_best_case)
                self.best_cases_iou.append(item)
    
    def add_worst_iou(self, item):
        if len(self.worst_cases_iou) < 5:
            self.worst_cases_iou.append(item)
        else:
            max_worst_case = max(self.worst_cases_iou, key=lambda x: (x['iou']))
            if (item['iou'] < max_worst_case['iou']):
                self.worst_cases_iou.remove(max_worst_case)
                self.worst_cases_iou.append(item)
    
    def add_best_confidence(self, item):
        if len(self.best_cases_sim) < 5:
            self.best_cases_sim.append(item)
        else:
            min_best_case = min(self.best_cases_sim, key=lambda x: (x['confidence']))
            if item['confidence'] > min_best_case['confidence']:
                self.best_cases_sim.remove(min_best_case)
                self.best_cases_sim.append(item)
    
    def add_worst_confidence(self, item):
        if len(self.worst_cases_sim) < 5:
            self.worst_cases_sim.append(item)
        else:
            max_worst_case = max(self.worst_cases_sim, key=lambda x: (x['confidence']))
            if (item['confidence'] < max_worst_case['confidence']):
                self.worst_cases_sim.remove(max_worst_case)
                self.worst_cases_sim.append(item)


    def update(self, iou, confidence,filename,bbox_e,bbox_gt):
        self.count += 1
        self.iou += iou
        if(iou >= self.threshold):
            self.correct_bboxes += 1
        self.semantic += confidence

        iteration = {
            'run_loc_acc': self.iou / self.count,
            'run_gro_acc': self.correct_bboxes / self.count,
            'run_sem_acc': self.semantic / self.count,
        }

        item = {
            'iou': iou,
            'confidence':confidence,
            'ground_truth': bbox_gt,
            'candidate': bbox_e,
            'path':filename
        }
        self.metrics.append([iteration])
        self.add_best_confidence(item)
        self.add_worst_confidence(item)
        self.add_best_iou(item)
        self.add_worst_iou(item)
        self.writer.add_scalar('Localization Loss', self.iou / self.count, self.count)
        self.writer.add_scalar('Grounding Accuracy',  self.correct_bboxes / self.count, self.count)
        self.writer.add_scalar('Semantic Similarity', self.semantic / self.count, self.count)

    def new_epoch(self):
        self.epoch += 1
        self.writer.add_scalar('Localization Loss', self.iou / self.count, self.epoch)
        self.writer.add_scalar('Grounding Accuracy',  self.correct_bboxes / self.count, self.epoch)
        self.writer.add_scalar('Semantic Similarity', self.semantic / self.count, self.epoch)
        self.epochs.append(self.metrics,self.best_cases_iou,self.best_cases_sim,self.worst_cases_iou,self.worst_cases_sim)
        self.reset()

    def __repr__(self):
        text = f"{self.name}: {self.avg:.8f}"
        return text
    
    def print_iteration(self):
        print(f"Localization accuracy = {self.iou / self.count}, Grounding Accuracy = {self.correct_bboxes / self.count}, Semantic Similarity = {self.semantic / self.count}")

    def get_best_iou_cases(self):
        return sorted(self.best_cases_iou, key=lambda x: x['iou'], reverse=True)

    def get_worst_iou_cases(self):
        return sorted(self.worst_cases_iou, key=lambda x: x['iou'])

    def get_best_pred_cases(self):
        return sorted(self.best_cases_sim, key=lambda x: x['confidence'], reverse=True)

    def get_worst_pred_cases(self):
        return sorted(self.worst_cases_sim, key=lambda x: x['confidence'])
    
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group["lr"]

In [9]:
with open("./refcocog/annotations/refs(umd).p", "rb") as fp:
  refs = pickle.load(fp)

# 'annotations' will be a dict object mapping the 'annotation_id' to the 'bbox' to make search faster
with open("./refcocog/annotations/instances.json", "rb") as fp:
  data = json.load(fp)
  annotations = dict(sorted({ann["id"]: ann["bbox"] for ann in data["annotations"]}.items()))

In [10]:
def getcaption(elem):
    li = []
    for e in elem["sentences"]:
        li.append(e['raw'])
    return li

In [13]:
from torchvision.ops import box_convert
class RefCOCOG(Dataset):
    """
    Args:
        The dataset will be the raw data wothput any tipe of preprocessing
        {
            'file_name': 
            'caption':
            'ann_id': needed to extract the relative bbox from the .json file
            'bbox': values are set like following:
                - x 
                - y
                - width 
                - height
        }
    """
    def __init__(self, refs, model, preprocess, annotations, split="train", device = 'cuda', count = 5000):
        
        self.clip_model, self.clip_preprocess = model, preprocess
        self.device = device
        self.images = []
        self.texts = []
        self.filepaths =[]
        self.gt = []
        self.cls = []
        temp = 0
        for elem in [d for d in refs if d["split"]==split]:
            
            # Retrieve Single image
            file_name = os.path.join("./refcocog/images/", f'{"_".join(elem["file_name"].split("_")[:3])}.jpg')
            image = Image.open(file_name)

            # Retrieve possible ground truth measures
            cls = elem['category_id']
            gt = annotations[elem['ann_id']]
            bbox_tnsor = torch.tensor(gt, device=self.device)
            new_bbox = box_convert(bbox_tnsor, 'xywh', 'xyxy')
            # Get all texts related to the picture
            sentences = elem['sentences']
            # for i in sentences:
            self.texts.append(clip.tokenize(sentences[0]['raw']))
            self.images.append(self.clip_preprocess(image))
            self.gt.append(new_bbox)
            self.cls.append(cls)
            self.filepaths.append(file_name)
            # temp += 1
            # if (temp > count):
            #     break

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        images = self.images[idx]
        gt = self.gt[idx]
        cls = self.cls[idx]
        filename = self.filepaths[idx]
        return text, images, gt, cls, filename

    def __call__(self, idx):
        print(json.dumps(self.dataset[idx], indent=4))


In [14]:
class RefCOCOG_noproc(Dataset):
    """
    Args:
        The dataset will be the raw data wothput any tipe of preprocessing
        {
            'file_name': 
            'caption':
            'ann_id': needed to extract the relative bbox from the .json file
            'bbox': values are set like following:
                - x 
                - y
                - width 
                - height
        }
    """
    def __init__(self, refs, model, preprocess, annotations, split="train", device = 'cuda', count = 31):
        
        self.clip_model, self.clip_preprocess = model, preprocess
        self.device = device
        #self.images = []
        self.texts = []
        self.filepaths =[]
        self.gt = []
        self.cls = []
        temp = 0
        for elem in [d for d in refs if d["split"]==split]:
            
            # Retrieve Single image
            file_name = os.path.join("./refcocog/images/", f'{"_".join(elem["file_name"].split("_")[:3])}.jpg')
            #image = Image.open(file_name)

            # Retrieve possible ground truth measures
            cls = elem['category_id']
            gt = annotations[elem['ann_id']]
            bbox_tnsor = torch.tensor(gt, device=self.device)
            new_bbox = box_convert(bbox_tnsor, 'xywh', 'xyxy')
            # Get all texts related to the picture
            sentences = elem['sentences']
            # for i in sentences:
            self.texts.append(clip.tokenize(sentences[0]['raw']))
            #self.images.append(self.clip_preprocess(image))
            self.gt.append(new_bbox)
            self.cls.append(cls)
            self.filepaths.append(file_name)
            temp += 1
            if (temp > count):
                break

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        #images = self.images[idx]
        gt = self.gt[idx]
        cls = self.cls[idx]
        filename = self.filepaths[idx]
        return text, gt, cls, filename#, images

    def __call__(self, idx):
        print(json.dumps(self.dataset[idx], indent=4))


In [10]:
# def pad_image(image):
#     """
#     Performs bottom-right padding of the original image to 640x640 (max size of images in the dataset).
#     Bottom-right padding prevents corruption of bounding boxes.

#     ### Arguments
#     image: a PIL.Image to transform
#     """
#     padded_width, padded_height = 640, 640
#     original_height, original_width = image.shape[:2]
#     bottom_padding = padded_height - original_height
#     right_padding = padded_width - original_width
#     top_padding = 0
#     left_padding = 0
    
#     padded_image = cv2.copyMakeBorder(image, top_padding, bottom_padding, left_padding, right_padding, cv2.BORDER_CONSTANT, value=[0, 0, 0])

#     return padded_image 

# def collate_fn(batch):
#     images = []
#     data = {}

#     #Stores all images in a list
#     for sample in batch:
#         text = sample[0]
#         images = sample[0]
#         gt = sample[0]
#         text = sample[0]
#         image = cv2.imread(sample["file_name"], 3)
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#         image = pad_image(image=image)

#         y_ = image.shape[0]
#         x_ = image.shape[1]


#         images.append(transform(image))

#         data['raw'] = sample['raw']
#         x,y,z,c = sample['bbox'][0:4]
#         y1=y 
#         x1=x 
#         y2=(y + c)
#         x2=(x + z)

#         data['bbox'] = [x,y,x2,y2]
#         data['filename'] = sample["file_name"]
            
#     images = torch.stack(images, dim=0)
#     """
#     for key in batch[0].keys():
#         #if key != "file_name":
#         #    data[key] = [sample[key] for sample in batch]
#         data[key] = [sample[key] for sample in batch]
#         if( key == 'bbox'):
#             x,y,z,c = sample[key][0:4]
#             y1=y 
#             x1=x 
#             y2=(y + z)
#             x2=(x + c)
#     """
#     return images, data

# transform = transforms.Compose([
#     transforms.ToTensor(),
# ])

In [15]:
# create dataset and dataloader
print("----------------------Processing train split----------------------------")
dataset_train = RefCOCOG_noproc(refs, clip_model, clip_preprocess, annotations, split="train")
dataloader_train = DataLoader(dataset_train, batch_size=16)
len_train = len(dataset_train)
print(f"Numero esempi in train = {len_train}")

print("----------------------Processing test split-----------------------------")
dataset_test = RefCOCOG_noproc(refs, clip_model, clip_preprocess, annotations, split="test")
dataloader_test = DataLoader(dataset_test, batch_size=16)
len_test = len(dataset_test)
print(f"Numero esempi in train = {len_test}")

print("----------------------Processing eval split-----------------------------")
dataset_eval = RefCOCOG_noproc(refs, clip_model, clip_preprocess, annotations, split="val")
dataloader_eval = DataLoader(dataset_eval, batch_size=16)
len_eval = len(dataset_eval)
print(f"Numero esempi in eval = {len_eval}")

print("------------------------------------------------------------------------")


----------------------Processing train split----------------------------
Numero esempi in train = 32
----------------------Processing test split-----------------------------
Numero esempi in train = 32
----------------------Processing eval split-----------------------------
Numero esempi in eval = 32
------------------------------------------------------------------------


In [16]:
##########################################################
# YolottoClip - Just for Zero-Shotting the dataset
##########################################################
from  torch.cuda.amp import autocast
MOMENTUM = 0.1 #Default should be 3e-4
class ConvBNReLU(nn.Module):
    '''Module for the Conv-BN-ReLU tuple.'''

    def __init__(self, c_in, c_out, kernel_size, stride, padding, dilation):
        super(ConvBNReLU, self).__init__()
        self.conv = nn.Conv2d(
                c_in, c_out, kernel_size=kernel_size, stride=stride, 
                padding=padding, dilation=dilation, bias=False)
        self.bn = nn.SyncBatchNorm(c_out, momentum=MOMENTUM)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class External_attention(nn.Module):

    '''
    Arguments:
        c (int): The input and output channel number.
    '''
    def __init__(self, c):
        super(External_attention, self).__init__()
        
        self.conv1 = nn.Conv2d(c, c, 1) #Convolution to linear layer
        self.al4 = 0
        self.k = 64
        self.fc0 = ConvBNReLU(2048, 512, 3, 1, 1, 1)
        self.linear_0 = nn.Conv1d(c, self.k, 1, bias=False)
        self.norm_layer = nn.SyncBatchNorm(c, momentum=MOMENTUM)
        self.linear_1 = nn.Conv1d(self.k, c, 1, bias=False)
        self.linear_1.weight.data = self.linear_0.weight.data.permute(1, 0, 2)        
        self.fc1 = nn.Sequential(
            ConvBNReLU(512, 256, 3, 1, 1, 1),
            nn.Dropout2d(p=0.1))
        self.conv2 = nn.Sequential(
            nn.Conv2d(c, c, 1, bias=False),
            self.norm_layer)    
        self.fc2 = nn.Conv2d(256, 80, 1)   
        self.final_linear = nn.Sequential(
            nn.Linear(3920, 1024)
        )
        
        for m in self.modules():
            if isinstance(m, nn.Conv2d): # Kaiming Initialiaztion
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.Conv1d):# He Initialization
                n = m.kernel_size[0] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, _BatchNorm): # BatchNorm Initialization all setted to 1
                m.weight.data.fill_(1)
                if m.bias is not None:
                    m.bias.data.zero_()

    # We need to compute the tensor one example by another in order for the memory units to 
    # grasp possible correlations between the different examples
    def process_features(self, tensor, shift=1):
        cycled_tensor = torch.roll(tensor, shifts=shift, dims=0)
        outputs = []
        
        for i in range(cycled_tensor.shape[0]):
            # Extract each slice along the first dimension
            tensor_slice = cycled_tensor[i]
            
            # Pass the slice through the model's forward method
            output = self.forward(tensor_slice.unsqueeze(0))  # Unsqueeze to add the batch dimension back
            
            # Store the output
            outputs.append(output)
        
        # Combine the outputs back into a single tensor if needed
        combined_output = torch.cat(outputs, dim=0)
        
        return combined_output


    def forward(self, x):
        # print(x.shape)
        x = self.fc0(x)
        idn = x
        x = self.conv1(x)

        b, c, h, w = x.size()
        n = h*w
        x = x.view(b, c, n)   # b * c * n 

        attn = self.linear_0(x) # b, k, n
        attn = F.softmax(attn, dim=-1) # b, k, n

        attn = attn / (1e-9 + attn.sum(dim=1, keepdim=True)) #  # b, k, n
        x = self.linear_1(attn) # b, c, n

        x = x.view(b, c, h, w)
        x = self.conv2(x)
        x = x + idn
        x = F.relu(x)
        x = self.fc1(x)
        x = self.fc2(x)
        x = x.view(1, -1)
        # print(x.shape)
        x = self.final_linear(x)

        return x
    
class ExternalYoloClip(nn.Module):
    def __init__(self, clip_model, clip_preprocess, device = 'cuda'):
        super(ExternalYoloClip, self).__init__()
        #self.yolo = YOLO("yolov8n.pt").eval()
        self.EA = External_attention(512).to(device)
        self.clip_model, self.clip_preprocess = clip_model, clip_preprocess
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        self.processing = []
        self.similarities = []
        self.device = device

    def encode_image(self, image):
        # Encode the image using the CLIP model
        with torch.no_grad():
            image_features = self.clip_model.encode_image(image)

        return image_features
    
    def __get_boxes__(self):
        print('ciao')
    
    def preprocess_images(self, cropped_images):
        # preprocess with CLIP each cropped PIL image(converts each image in a image
        # of size [3,224,224])
        processing = []
        for image in cropped_images:
            processed_img = self.clip_preprocess(image).to(self.device)
            # proc = self.clip_model.encode_image(processed_img)
            processing.append(processed_img)
        preprocessed = torch.tensor(np.stack(processing))
        # preprocessed = torch.stack(processing).to(self.device) # return a single tensor
        return preprocessed


    def encode_text(self, text):
        with torch.no_grad():
            text_features = self.clip_model.encode_text(text)
        return text_features
    
    def hook_fn(self, module, input, output):
        self.al = output

    def forward(self, images, text):
        # print(images.shape)
        #bl = 0 # input of layer4 of CLIP's ResNet
        #print(len(images))
        hook_handle = clip_model.visual.layer4.register_forward_hook(self.hook_fn) # handle to retrieve output
        image_features = self.encode_image(images)
        # print(image_features)
        # print(70*'-')
        # # print(self.al.shape)
        with autocast():
            image_features = self.EA.process_features(self.al)
        # print(image_features)
        # print(70*'-')
        # print(image_features.shape)
        # print(70*'-')
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = self.encode_text(text)
        # temp = []
        # temp.append(text_features)
        # temp.append(text_features)
        # temp.append(text_features)
        # text_features2 = torch.cat(temp, dim=0)
        # print(text_features)
        # print(70*'-')
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        # text_features2 = text_features2 / text_features2.norm(dim=-1, keepdim=True)
        logit_scale = self.logit_scale.exp()
        # normalized featuresim
        similarity = (image_features @ text_features.t())
        # print(similarity.cpu())
        # similarity = (image_features @ text_features2.t())
        # print(similarity.cpu())
        scaled_similarity = logit_scale * similarity
        logits_per_image = scaled_similarity
        logits_per_text = scaled_similarity.t()

        # print(text_features.shape)
        # # cosine similarity as logits
        # logits_per_image = logit_scale * (image_features @ text_features.t())
        # print("Logits = " + str(logits_per_image))
        # logits_per_text = logits_per_image.t()

        # shape = [global_batch_size, global_batch_size]
        # hook_handle.remove()

        return logits_per_image, logits_per_text
    
    def evaluator(self, crop_images, text, filepath):
        self.eval()
        best_score = 0
        best_bbox = None
        candidates = []
        images = []

        ex_bbox, cls = self.infer_bboxes(images)

        for bbox in self.infer_bboxes(ex_bbox):
            temp = cv2.imread(filepath)
            image = np.zeros((temp.shape[0], temp.shape[1], temp.shape[2]), dtype=np.uint8)
            image[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])] = temp[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])]
            image = self.clip_preprocess(Image.fromarray(image)).unsqueeze(0).to(device)
            images.append(image)
        
        li,lt = self.model(images,text)
        print(li)


In [27]:
model = ExternalYoloClip(clip_model= clip_model, clip_preprocess=clip_preprocess)

In [ ]:
#model.load_state_dict(torch.load())

In [96]:
def infer_bboxes(image_path, yolo):
    yolo = yolo
    #print(image_path)
    results = yolo(image_path, verbose=False)
    # print(results[0])
    bboxes = results[0].boxes.xyxy
    #cls = results[0].boxes.cls
    return bboxes

def get_candidate_crop(text,path,yolo):
    images = []
    best_score = 0
    best_bbox = None

    for bbox in infer_bboxes(path, yolo):
        temp = cv2.imread(path)
        image = np.zeros((temp.shape[0], temp.shape[1], temp.shape[2]), dtype=np.uint8)
        image[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])] = temp[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])]
        image = clip_preprocess(Image.fromarray(image)).to(device)
        print(image.shape)
        images.append(image)
        imagesa = torch.stack(images)
        print(imagesa.shape)

        with torch.no_grad():
                logits_per_image, logits_per_text = model(imagesa[0], text)
                matching_score = logits_per_text.cpu().numpy()[0]

        if matching_score > best_score:
                best_score = matching_score
                best_bbox = bbox
    return best_score, best_bbox


In [97]:
def train_Pipeline(model):
    model.clip_model.train()
    model.EA.train()

def eval_Pipeline(model):
    model.clip_model.eval()
    model.EA.eval()

In [98]:
cumulative_accuracy = 0.0
cumulative_loss = 0.0
overall = 0
errors = 0
yolo = YOLO('yolov8n.pt')
cumulative_iou = 0.0
cumulative_recall = 0.0
cumulative_sim = 0.0
iou_threshold = 0.5
correct_bboxes = 0


eval_loop = tqdm(dataloader_eval, position=0, leave=True)
eval_Pipeline(model)
with torch.no_grad():
    for _,data in enumerate(eval_loop):
        scores = []
        bboxes = []

        texts = data[0]
        texts = texts.squeeze(1).to(device)
        #images = data[1].to(device)
        gts = data[1]
        clss = data[2]
        filename = data[3]
        images = []
        
        for x in filename:
            temp = Image.open(x)
            image = clip_preprocess(temp).to(device)
            print(image.shape)
            images.append(image)

        fin = torch.stack(images)
        batch = len(images)

        for i in range(batch):
            score, ebbox = get_candidate_crop(texts[i], filename[i],yolo)
            # scores.append(score)
            # bboxes.append(ebbox)
            iou = compute_iou(ebbox, gts[i])
            cumulative_iou += iou
            if(iou > iou_threshold):
                #compare_candidate_bbox(filename,bbox_e, bbox_gt)
                correct_bboxes += 1
            similarity = score / 100
            cumulative_sim += score

            loc_acc = cumulative_iou / overall
            ga = cumulative_accuracy / overall
            semsim = cumulative_sim / overall 

        
        overall += batch
        eval_loop.set_description(f"Localization accuracy = {loc_acc}, Grounding Accuracy = {ga}, Semantic Similarity = {semsim}")



  0%|          | 0/2 [00:00<?, ?it/s]

torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
torch.Size([1, 3, 224, 224])


ValueError: expected 4D input (got 3D input)

In [ ]:
model.evaluator(image,bbox,temp)sda

New https://pypi.org/project/ultralytics/8.2.82 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.57 🚀 Python-3.9.19 torch-2.3.0+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 5931MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=coco.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False,

100%|██████████| 169M/169M [00:43<00:00, 4.04MB/s] 
Unzipping /home/matea/Documents/VisualGrounding/DeepLearning/Code/datasets/coco2017labels-segments.zip to /home/matea/Documents/VisualGrounding/DeepLearning/Code/datasets/coco...: 100%|██████████| 122232/122232 [00:11<00:00, 10222.74file/s]

KeyboardInterrupt: 

In [ ]:
NUM_EPOCHS = 15
count = 0
iou_threshold = 0.5
loss_meter = MetricMeter(name = 'TrainingEA', threshold=iou_threshold)

running_loc_acc = 0.0
correct_bboxes = 0
overall = 0
semantic = 0
semantic_similarity = 0
running_ga = 0
sem = 0
loc_acc = 0
model = ExternalYoloClip(clip_model,clip_preprocess,device='cuda')

In [ ]:
loss_meter.reset()

In [ ]:
lr = 0.0001
# wd = 0.002 # best run so far
wd = 0.001
alpha = 1 # to decrease lr over time

In [ ]:
cost = nn.CrossEntropyLoss()
optimizer = torch.optim.Adadelta(model.parameters(), lr=lr, weight_decay = wd)
optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=wd)

In [ ]:
def freeze_layers(model):
    for param in model.clip_model.parameters():
        param.requires_grad = False

def unfreeze_layers(model):
    for param in model.clip_model.parameters():
        param.requires_grad = True

In [ ]:
cumulative_accuracy = 0.0
cumulative_loss = 0.0
overall = 0
errors = 0

test_cumulative_accuracy = 0.0
test_cumulative_loss = 0.0
test_overall = 0
test_errors = 0

for i in range(NUM_EPOCHS):
    loop = tqdm(dataloader_train, position=0, leave=True)
    test_loop = tqdm(dataloader_test, position=0, leave=True)
    train_Pipeline(model)
    for _,data in enumerate(loop):
        try:
        
            texts = data[0]
            texts = texts.squeeze(1).to(device)
            #images = data[1].to(device)
            gts = data[1]
            clss = data[2]
            filename = data[3]
            images = []
            
            for x in filename:
                temp = Image.open(x)
                image = clip_preprocess(temp).to(device)
                images.append(image)

            images = torch.stack(images)
            optimizer.zero_grad()

            # Build Data for training pass

            # since confidence is directly how much bbox and text "resembles" each other 
            li, lt = model.forward(images,texts)

            # Construct the ground truth
            ground_truth = torch.arange(len(images),dtype=torch.long,device=device)
            img_loss = cost(li, ground_truth)
            desc_loss = cost(lt, ground_truth)
            loss = (img_loss + desc_loss)/2
            loss.backward()
            optimizer.step()

            # Keep track of loss and accuracy metrics to see epochs progress
            overall += 16 #batch_size
            cumulative_loss += loss.item()

            _, predicted = li.max(dim=1)
            cumulative_accuracy += predicted.eq(ground_truth).sum().item()
            loss = cumulative_loss / overall 
            acc = cumulative_accuracy / overall
        except:
            errors += 1
            print("diocan un altro " + str(errors))


        loop.set_description(f"Training Epoch {i} values => Loss Iter = {loss}, Accuracy = {acc}")
    
    if( i % 3 == 0):
        eval_Pipeline(model)
        with torch.no_grad():
            for _,data in enumerate(test_loop):
                texts = data[0]
                texts = texts.squeeze(1).to(device)
                #images = data[1].to(device)
                gts = data[1]
                clss = data[2]
                filename = data[3]
                images = []
                
                for x in filename:
                    temp = Image.open(x)
                    image = clip_preprocess(temp).to(device)
                    images.append(image)

                images = torch.stack(images)
                li, lt = model.forward(images,texts)
                ground_truth = torch.arange(len(images),dtype=torch.long,device=device)
                img_loss = cost(li, ground_truth)
                desc_loss = cost(lt, ground_truth)
                loss = (img_loss + desc_loss)/2
                overall += 16 #batch_size
                test_cumulative_loss += loss.item()
                _, predicted = li.max(dim=1)
                test_cumulative_accuracy += predicted.eq(ground_truth).sum().item()
                loss = test_cumulative_loss / overall 
                acc = test_cumulative_accuracy / overall
                test_loop.set_description(f"Testin Epoch {i}. Values => Loss Iter = {loss}, Accuracy = {acc}")


NameError: name 'NUM_EPOCHS' is not defined

In [ ]:
torch.save(model.state_dict(), 'saves/EA.pt')

In [ ]:
import gc

model.cpu()
del model
gc.collect()
torch.cuda.empty_cache()

# RPN

In [ ]:
class AnchorGenerator(torch.nn.Module):

  def __init__(self, scales, ratios, num_centers=7, spatial_dim=640, device=None):
    super().__init__()

    if device is None:
      self.device = "cuda" if torch.cuda.is_available () else "cuda"
    else:
      self.device = device

    anchorCenters = self.getAnchorCenters(num_centers, spatial_dim)
    self.anchors = self.getAnchorBoxes(anchorCenters, scales, ratios, spatial_dim)
    self.anchors_per_pixel = len(scales) * len(ratios)

  def forward(self, images, *arg):
    batch_size = len(images.image_sizes)
    return [self.anchors] * batch_size

  def __len__(self):
    return self.anchors_per_pixel

  '''
  Returns the anchor centers evenly spaced along the spatial dimension
  specified. The parameter 'num_centers' should correspond with the spatial
  dimension of the feature map.
  '''
  def getAnchorCenters(self, num_centers, spatial_dim):
      interval = math.floor(spatial_dim / num_centers)
      return torch.arange(0, spatial_dim - interval, interval)

  '''
  Starting from the given 'anchorCenters' this function generates all the
  possible anchor boxes that can be formed combining 'scales' and 'ratios'.

  Returns tensor [N, K]
  '''
  def getAnchorBoxes(self, anchorCenters, scales, ratios, spatial_dim):

      num_centers = anchorCenters.size(0)
      num_scales = scales.size(0)
      num_ratios = ratios.size(0)
      num_anchors_per_pixel = num_scales * num_ratios

      # compute the combinations of widths and heights according to the rations
      # and scales
      wh_combinations = torch.zeros((num_scales * num_ratios, 2), device=self.device)

      h_ratios = torch.sqrt(ratios)
      w_ratios = 1 / h_ratios

      i = 0
      for ratio_i in range(ratios.size(0)):
        for scale in scales:
          wh_combinations[i,0] = scale * w_ratios[ratio_i] # width
          wh_combinations[i,1] = scale * h_ratios[ratio_i] # height
          i += 1

      wh_combinations = wh_combinations.repeat(num_centers * num_centers,1)

      # compute the combinations of centers positions
      centers = torch.zeros((num_centers * num_centers, 2), device=self.device)
      i = 0
      for cy in anchorCenters:
        for cx in anchorCenters:
          centers[i,0] = cx
          centers[i,1] = cy
          i += 1

      anchors = centers.repeat_interleave(repeats=num_anchors_per_pixel, dim=0)
      anchors = torch.cat([anchors, wh_combinations], dim=1)
      anchors_xyxy = box_convert(anchors, 'cxcywh', 'xyxy')

      return anchors_xyxy.round()

In [ ]:
scales = torch.tensor([32, 64, 128, 256, 512])
ratios = torch.tensor([0.5, 1.0, 2.0])

anchor_generator = AnchorGenerator(scales, ratios, num_centers=7, spatial_dim=640)

In [ ]:
import torchvision.models.detection.rpn as rpn
class RPNHead(torch.nn.Module):

    def __init__(self, feature_map_channels, anchors_per_pixel):
        super().__init__()

        # The convolutions used to calculate the objectness and the regression offsets
        self.conv = torchvision.ops.Conv2dNormActivation(feature_map_channels, feature_map_channels, kernel_size=3, activation_layer=torch.nn.ReLU, norm_layer=None)
        self.cls_logits = torch.nn.Conv2d(feature_map_channels, anchors_per_pixel, kernel_size=1, stride=1)
        self.reg_offsets = torch.nn.Conv2d(feature_map_channels, anchors_per_pixel * 4, kernel_size=1, stride=1)

        # Convolutions initialization taken from the original pytorch implementation
        for layer in self.modules():
            if isinstance(layer, torch.nn.Conv2d):
                torch.nn.init.normal_(layer.weight, std=0.01)
                if layer.bias is not None:
                    #layer.bias.data = layer.bias.data.float()
                    torch.nn.init.constant_(layer.bias, 0)

        

    def forward(self, feature_maps):
        feature_map = feature_maps[0]

        activated_fm = self.conv(feature_map)
        cls_logits = self.cls_logits(activated_fm)
        bbox_reg = self.reg_offsets(activated_fm)

        return [cls_logits], [bbox_reg]

In [ ]:
class ImageList:
    def __init__(self, image_sizes):
        self.image_sizes = image_sizes

def getLoss(cls_loss, reg_loss, lambda_rpn=1):
    return cls_loss + lambda_rpn * reg_loss

def computeSumIou(gtBoxes, predictedBoxes):
  iouMatrix = torchvision.ops.box_iou(gtBoxes, predictedBoxes).diag()
  return iouMatrix.sum()

In [ ]:
##########################################################
# YolottoClip - Just for Zero-Shotting the dataset
##########################################################
# from torch.amp import autocast
MOMENTUM = 0.1 #Default should be 3e-4
class ConvBNReLU(nn.Module):
    '''Module for the Conv-BN-ReLU tuple.'''

    def __init__(self, c_in, c_out, kernel_size, stride, padding, dilation):
        super(ConvBNReLU, self).__init__()
        self.conv = nn.Conv2d(
                c_in, c_out, kernel_size=kernel_size, stride=stride, 
                padding=padding, dilation=dilation, bias=False)
        self.bn = nn.SyncBatchNorm(c_out, momentum=MOMENTUM)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class External_attention(nn.Module):

    '''
    Arguments:
        c (int): The input and output channel number.
    '''
    def __init__(self, c):
        super(External_attention, self).__init__()
        
        self.conv1 = nn.Conv2d(c, c, 1) #Convolution to linear layer
        self.al4 = 0
        self.k = 64
        self.fc0 = ConvBNReLU(2048, 512, 3, 1, 1, 1)
        self.linear_0 = nn.Conv1d(c, self.k, 1, bias=False)
        self.norm_layer = nn.SyncBatchNorm(c, momentum=MOMENTUM)
        self.linear_1 = nn.Conv1d(self.k, c, 1, bias=False)
        self.linear_1.weight.data = self.linear_0.weight.data.permute(1, 0, 2)        
        self.fc1 = nn.Sequential(
            ConvBNReLU(512, 256, 3, 1, 1, 1),
            nn.Dropout2d(p=0.1))
        self.conv2 = nn.Sequential(
            nn.Conv2d(c, c, 1, bias=False),
            self.norm_layer)    
        self.fc2 = nn.Conv2d(256, 80, 1)   
        self.final_linear = nn.Sequential(
            nn.Linear(3920, 1024)
        )
        
        for m in self.modules():
            if isinstance(m, nn.Conv2d): # Kaiming Initialiaztion
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.Conv1d):# He Initialization
                n = m.kernel_size[0] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, _BatchNorm): # BatchNorm Initialization all setted to 1
                m.weight.data.fill_(1)
                if m.bias is not None:
                    m.bias.data.zero_()

    def process_features(self, tensor, shift=1):
        cycled_tensor = torch.roll(tensor, shifts=shift, dims=0)
        outputs = []
        
        for i in range(cycled_tensor.shape[0]):
            # Extract each slice along the first dimension
            tensor_slice = cycled_tensor[i]
            
            # Pass the slice through the model's forward method
            output = self.forward(tensor_slice.unsqueeze(0))  # Unsqueeze to add the batch dimension back
            
            # Store the output
            outputs.append(output)
        
        # Combine the outputs back into a single tensor if needed
        combined_output = torch.cat(outputs, dim=0)
        
        return combined_output


    def forward(self, x):
        # print(x.shape)
        x = self.fc0(x)
        idn = x
        x = self.conv1(x)

        b, c, h, w = x.size()
        n = h*w
        x = x.view(b, c, n)   # b * c * n 

        attn = self.linear_0(x) # b, k, n
        attn = F.softmax(attn, dim=-1) # b, k, n

        attn = attn / (1e-9 + attn.sum(dim=1, keepdim=True)) #  # b, k, n
        x = self.linear_1(attn) # b, c, n

        x = x.view(b, c, h, w)
        x = self.conv2(x)
        x = x + idn
        x = F.relu(x)
        x = self.fc1(x)
        x = self.fc2(x)
        x = x.view(1, -1)
        # print(x.shape)
        x = self.final_linear(x)

        return x
    
class ExternalRPNClip(nn.Module):
    def __init__(self, clip_model, clip_preprocess, anchors_scales, anchor_ratios, feature_map_channels=2048, device = 'cuda'):
        super(ExternalRPNClip, self).__init__()
        self.yolo = YOLO("yolov8n.pt")
        self.EA = External_attention(512).to(device)
        self.clip_model, self.clip_preprocess = clip_model, clip_preprocess
        
        # Prepare the Region proposal network with it's anchors and the RPN head
        self.anchor_generator = AnchorGenerator(anchors_scales, anchor_ratios)
        self.rpn_head = RPNHead(feature_map_channels, len(self.anchor_generator))

        rpn_pre_post_nms_top_n = {"training": 200, "testing": 100}
        self.rpn_wrapper = rpn.RegionProposalNetwork(
            self.anchor_generator, self.rpn_head,
            fg_iou_thresh=0.7,
            bg_iou_thresh=0.2,          # anchor boxes with a IOU < 0.2 are considered negative
            batch_size_per_image=256,   # for each image 256 anchors are sampled
            positive_fraction=0.5,      # out of the 256 anchors half are positive and half negative
            pre_nms_top_n=rpn_pre_post_nms_top_n,
            post_nms_top_n=rpn_pre_post_nms_top_n,
            nms_thresh=0.7              # threshold for Non maximum suppression
        ).to(device)
        
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        self.processing = []
        self.similarities = []
        self.device = device

    def infer_bboxes(self, image_path):
        results = self.yolo(image_path, verbose=False)
        # print(results[0])
        bboxes = results[0].boxes.xyxy
        cls = results[0].boxes.cls
        return bboxes,cls

    def encode_image(self, image):
        # Encode the image using the CLIP model
        with torch.no_grad():
            image_features = self.clip_model.encode_image(image)

        return image_features
    
    def preprocess_images(self, cropped_images):
        # preprocess with CLIP each cropped PIL image(converts each image in a image
        # of size [3,224,224])
        processing = []
        for image in cropped_images:
            processed_img = self.clip_preprocess(image).to(self.device)
            # proc = self.clip_model.encode_image(processed_img)
            processing.append(processed_img)
        preprocessed = torch.tensor(np.stack(processing))
        # preprocessed = torch.stack(processing).to(self.device) # return a single tensor
        return preprocessed


    def encode_text(self, text):
        with torch.no_grad():
            text_features = self.clip_model.encode_text(text)
        return text_features
    
    def hook_fn(self, module, input, output):
        self.al = output
        self.al.to(device)

    def forward(self, images, text, gtBoxes, len_im):
        # print(images.shape)
        #print(images)
        #bl = 0 # input of layer4 of CLIP's ResNet
        hook_handle = clip_model.visual.layer4.register_forward_hook(self.hook_fn) # handle to retrieve output
        image_features = self.encode_image(images)
        # print(70*'-')
        # # print(self.al.shape)
        with torch.autocast(device_type="cuda"):
            image_features = self.EA.process_features(self.al)
        # print(image_features)
        # print(70*'-')
        # Region Proposal Network
        ground_truth_boxes = [{"boxes": gtBoxes[batch_i].unsqueeze(0)} for batch_i in range(len_im)]
        image_sizes = ImageList([(640,640)] * len_im)   #batch_size = 16
        
        #print(ground_truth_boxes)

        with torch.autocast(device_type="cuda"):
            proposals_RPN, losses_RPN = self.rpn_wrapper(image_sizes, {"0": self.al}, ground_truth_boxes)
        #print(losses_RPN)
        
         #print("RPN: ", rpn_result)
        # print(70*'-')
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = self.encode_text(text)
        # temp = []
        # temp.append(text_features)
        # temp.append(text_features)
        # temp.append(text_features)
        # text_features2 = torch.cat(temp, dim=0)
        # print(text_features)
        # print(70*'-')
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        # text_features2 = text_features2 / text_features2.norm(dim=-1, keepdim=True)
        logit_scale = self.logit_scale.exp()

        # Conversion to float32
        image_features = image_features.float()
        text_features = text_features.float()
        
        # normalized featuresim
        similarity = (image_features @ text_features.t())
        # print(similarity.cpu())
        # similarity = (image_features @ text_features2.t())
        # print(similarity.cpu())
        scaled_similarity = logit_scale * similarity
        logits_per_image = scaled_similarity
        logits_per_text = scaled_similarity.t()

        # print(text_features.shape)
        # # cosine similarity as logits
        # logits_per_image = logit_scale * (image_features @ text_features.t())
        # print("Logits = " + str(logits_per_image))
        # logits_per_text = logits_per_image.t()

        # shape = [global_batch_size, global_batch_size]
        # hook_handle.remove()

        return logits_per_image, logits_per_text, proposals_RPN, losses_RPN
    

In [ ]:
NUM_EPOCHS = 10
count = 0
iou_threshold = 0.5
loss_meter = MetricMeter(name = 'TrainingEA', threshold=iou_threshold)

running_loc_acc = 0.0
correct_bboxes = 0
overall = 0
semantic = 0
semantic_similarity = 0
running_ga = 0
sem = 0
loc_acc = 0
lr = 0.0001
wd = 0.002
alpha = 1 # to decrease lr over time
model = ExternalRPNClip(clip_model,clip_preprocess, anchors_scales=scales, anchor_ratios=ratios,device='cuda')

In [ ]:
def train_Pipeline(model):
    model.clip_model.train()
    model.EA.train()
    model.rpn_head.train()
    model.rpn_wrapper.train()

def eval_Pipeline(model):
    model.clip_model.eval()
    model.EA.eval()
    model.rpn_head.eval()
    model.rpn_wrapper.eval()

In [ ]:
cost = nn.CrossEntropyLoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=wd)

In [ ]:
cumulative_accuracy = 0.0
cumulative_loss = 0.0
overall = 0
errors = 0

test_cumulative_accuracy = 0.0
test_cumulative_loss = 0.0
test_overall = 0
test_errors = 0

for i in range(NUM_EPOCHS):
    loop = tqdm(dataloader_train, position=0, leave=True)
    test_loop = tqdm(dataloader_test, position=0, leave=True)
    train_Pipeline(model)
    for _,data in enumerate(loop):
        try:
            
            texts = data[0]
            texts = texts.squeeze(1).to(device)
            #images = data[1].to(device)
            gts = data[1]
            clss = data[2]
            filename = data[3]
            images = []
            
            for x in filename:
                temp = Image.open(x)
                image = clip_preprocess(temp).to(device)
                images.append(image)

            images = torch.stack(images)
            optimizer.zero_grad()
            len_im = len(images)# last pass could contain less than 16 images

            # Build Data for training pass
            # since confidence is directly how much bbox and text "resembles" each other 
            li, lt, proposals_RPN, losses_RPN = model.forward(images, texts, gts, len_im)

            # Region Proposal Network
            # Calculate the loss
            loss_RPN = getLoss(losses_RPN["loss_objectness"], losses_RPN["loss_rpn_box_reg"])
            #avg_loss += loss.item()

            # calculate the IOU
            best_proposals = torch.zeros((len_im, 4), device="cuda")
            for j in range(len_im):
                # The proposal with the highest score(the first) is the final one
                best_proposals[j,:] = proposals_RPN[j][0]

            cumulative_accuracy += computeSumIou(gts, best_proposals).item()



            # Construct the ground truth
            ground_truth = torch.arange(len(images),dtype=torch.long,device=device)
            img_loss = cost(li, ground_truth)
            desc_loss = cost(lt, ground_truth)
            loss = (img_loss + desc_loss)/2
            loss += loss_RPN
            loss.backward()
            optimizer.step()

            # Keep track of loss and accuracy metrics to see epochs progress
            overall += len_im #batch_size
            cumulative_loss += loss.item()

            _, predicted = li.max(dim=1)
            cumulative_accuracy += predicted.eq(ground_truth).sum().item()
            loss = cumulative_loss / overall 
            acc = cumulative_accuracy / overall
            #clip.model.convert_weights(model)
        except:
            print('errore nel training')

        loop.set_description(f"Training \Epoch{i}\ Values => Loss Iter = {loss}, Accuracy = {acc}")
    
    if( i % 3 == 0):
        eval_Pipeline(model)
        with torch.no_grad():
            for _,data in enumerate(test_loop):
                try:
                    texts = data[0]
                    texts = texts.squeeze(1).to(device)
                    #images = data[1].to(device)
                    gts = data[1]
                    clss = data[2]
                    filename = data[3]
                    images = []
                    for x in filename:
                        temp = Image.open(x)
                        image = clip_preprocess(temp).to(device)
                        images.append(image)

                    images = torch.stack(images)
                    #optimizer.zero_grad()
                    len_im = len(images)# last pass could contain less than 16 images

                    # Build Data for training pass
                    # since confidence is directly how much bbox and text "resembles" each other 
                    li, lt, proposals_RPN, losses_RPN = model.forward(images, texts, gts, len_im)

                    print(losses_RPN)
                    # Region Proposal Network
                    # Calculate the loss
                    loss_RPN = getLoss(losses_RPN["loss_objectness"], losses_RPN["loss_rpn_box_reg"])
                    #avg_loss += loss.item()

                    # calculate the IOU
                    best_proposals = torch.zeros((len_im, 4), device="cuda")
                    for j in range(len_im):
                        # The proposal with the highest score(the first) is the final one
                        best_proposals[j,:] = proposals_RPN[j][0]

                    cumulative_accuracy += computeSumIou(gts, best_proposals).item()



                    # Construct the ground truth
                    ground_truth = torch.arange(len(images),dtype=torch.long,device=device)
                    img_loss = cost(li, ground_truth)
                    desc_loss = cost(lt, ground_truth)
                    loss = (img_loss + desc_loss)/2
                    loss += loss_RPN
                    # loss.backward()
                    # optimizer.step()

                    # Keep track of loss and accuracy metrics to see epochs progress
                    overall += len_im #batch_size
                    cumulative_loss += loss.item()

                    _, predicted = li.max(dim=1)
                    cumulative_accuracy += predicted.eq(ground_truth).sum().item()
                    loss = cumulative_loss / overall 
                    acc = cumulative_accuracy / overall
                    #clip.model.convert_weights(model)
                except:
                    print('errore nel testing')

                    test_loop.set_description(f"Testing \Epoch{i}\ Values => Loss Iter = {loss}, Accuracy = {acc}")

  0%|          | 0/5032 [00:00<?, ?it/s]

  0%|          | 0/601 [00:00<?, ?it/s]

errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training


  0%|          | 0/5032 [00:00<?, ?it/s]

  0%|          | 0/601 [00:00<?, ?it/s]

...

  0%|          | 0/601 [00:00<?, ?it/s]

errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training


  0%|          | 0/5032 [00:00<?, ?it/s]

  0%|          | 0/601 [00:00<?, ?it/s]

errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training
errore nel training


In [ ]:
torch.save(model.state_dict(), 'saves/EARPN.pt')